In [103]:
!pip install pyscf
!pip install pyscf-semiempirical




In [104]:
from pyscf import gto, scf, semiempirical
import numpy as np
from pyscf.data.nist import HARTREE2EV


np.set_printoptions(precision = 4 , suppress = True)

In [106]:
def show(name, labels, mf, orbitals):
    #Число МО, энергии занятых МО и коэффициенты разложения выбранных МО по АО.
    nocc = int(np.sum(mf.mo_occ > 0))
    print(f"\n{name}")
    print("Число орбиталей:", len(mf.mo_energy))
    print("Энергии занятых МО (Eh):", np.round(mf.mo_energy[:nocc], 5)) #взяли список энергий всех орбиталей, отрезали {nocc} значений, так как нам нужны энергии только оккупированных
    for i in orbitals:                      
        print(f"Состав МО{i+1} (коэффициенты при АО):")
        for lab, c in zip(labels, mf.mo_coeff[:, i]): #использую zip чтобы название АО и ее коэфицент были в одном кортеже. но можно было и подругому через range(len(labels))
            print(f"   {lab:12s} {c: .5f}")

 
 


In [107]:
water = """
O 0 0 0.11779 
H 0 0.755453 -0.471161
H 0 -0.755453 -0.471161
"""

for basis in ["sto-3g", "sto-6g", "def2-svp", "aug-cc-pvdz"]:
    mol = gto.M(atom=water, basis=basis, verbose=0)
    mf = scf.RHF(mol).run()
    show(f"H2O HF/{basis}", mol.ao_labels(), mf, [0])



H2O HF/sto-3g
Число орбиталей: 7
Энергии занятых МО (Eh): [-20.2424  -1.2685  -0.6169  -0.4539  -0.3915]
Состав МО1 (коэффициенты при АО):
   0 O 1s        0.99413
   0 O 2s        0.02654
   0 O 2px       0.00000
   0 O 2py       0.00000
   0 O 2pz      -0.00435
   1 H 1s       -0.00596
   2 H 1s       -0.00596

H2O HF/sto-6g
Число орбиталей: 7
Энергии занятых МО (Eh): [-20.5048  -1.2763  -0.6202  -0.4602  -0.3977]
Состав МО1 (коэффициенты при АО):
   0 O 1s        0.99664
   0 O 2s        0.01597
   0 O 2px      -0.00000
   0 O 2py       0.00000
   0 O 2pz      -0.00325
   1 H 1s       -0.00376
   2 H 1s       -0.00376

H2O HF/def2-svp
Число орбиталей: 24
Энергии занятых МО (Eh): [-20.5461  -1.3189  -0.7026  -0.5689  -0.4982]
Состав МО1 (коэффициенты при АО):
   0 O 1s        0.99033
   0 O 2s       -0.03332
   0 O 3s        0.01040
   0 O 2px       0.00000
   0 O 2py      -0.00000
   0 O 2pz       0.00279
   0 O 3px       0.00000
   0 O 3py       0.00000
   0 O 3pz      -0.00016
  

STO-3G and STO-6G - базисные наборы Попла, они отличаются только количеством гауссовых орбиталей(3 и 6 соответсвенно), используемых для описания одной слейтеровской (так как она физически лучше описывает АО). Следовательно количество базисных функций в них одинаково, так как мы находим коэфиценты для аппроксимированных слейтеровских функций, а не гауссовых. STO-6G будет точнее, так как лучше fit.

In [108]:
a = 0.629
ch4 = (f"C 0 0 0; H {a} {a} {a}; H -{a} -{a} {a}; "
       f"H -{a} {a} -{a}; H {a} -{a} -{a}")
mol = gto.M(atom=ch4, basis="sto-3g", verbose=1)
mf = scf.RHF(mol).run()
mf.energy_tot()
show("CH4 HF/sto-3g", mol.ao_labels(), mf, [0, 1])



CH4 HF/sto-3g
Число орбиталей: 9
Энергии занятых МО (Eh): [-11.0303  -0.9088  -0.518   -0.518   -0.518 ]
Состав МО1 (коэффициенты при АО):
   0 C 1s        0.99196
   0 C 2s        0.03805
   0 C 2px       0.00000
   0 C 2py       0.00000
   0 C 2pz       0.00000
   1 H 1s       -0.00693
   2 H 1s       -0.00693
   3 H 1s       -0.00693
   4 H 1s       -0.00693
Состав МО2 (коэффициенты при АО):
   0 C 1s       -0.22119
   0 C 2s        0.62938
   0 C 2px       0.00000
   0 C 2py       0.00000
   0 C 2pz      -0.00000
   1 H 1s        0.18075
   2 H 1s        0.18075
   3 H 1s        0.18075
   4 H 1s        0.18075


In [110]:
mol = gto.M(atom=ch4, verbose=0)
mf = semiempirical.MINDO3(mol)
mf.kernel(dm0=np.zeros((8, 8)))  #это прищлось делать, так как semiempirical выдавал ошибку, связанную со своим кодом, с этой строкой помог ИИ
labels = ["C 2s", "C 2px", "C 2py", "C 2pz", "H1 1s", "H2 1s", "H3 1s", "H4 1s"] #ручками назывем орбитали, выбраны такие, так как mindo3 - валентный метод
show("CH4 MINDO3", labels, mf, [0])



CH4 MINDO3
Число орбиталей: 8
Энергии занятых МО (Eh): [-1.0027 -0.4903 -0.4903 -0.4903]
Состав МО1 (коэффициенты при АО):
   C 2s          0.79304
   C 2px         0.00000
   C 2py         0.00000
   C 2pz         0.00000
   H1 1s         0.30458
   H2 1s         0.30458
   H3 1s         0.30458
   H4 1s         0.30458


У MINDO3 меньше  орбиталей, отсутствует функция похожая на C1s орбиталь(то есть нижней по энергии), так как mindo3 это метод, рассматривающий только валентные электроны, в которые C1s не входит. Остальные энергии относительно близки, в обоих расчетах видно a1 + t2 расщепление. 